# Kaggle T4 ×2: xác nhận hai smoke run song song

Một checkout, shared HF cache, **một CSV tracker**, hai process/GPU riêng.
GPU 0: exp-003 XLM-R, run-014. GPU 1: exp-007 mDeBERTa, run-015.
Mỗi run dùng original head, seed 42, batch 8, 2 epoch, 256 train/64 validation.
Đây là diagnostic, không phải kết quả luận văn và không gọi evaluate test.

Bật **GPU T4 ×2** và **Internet**. Push/upload code mới có khóa tracker FileLock trước
khi dùng notebook. Chạy cell theo thứ tự; không chạy lại cell launch khi process còn sống.
Hai T4 không gộp bộ nhớ: mỗi model phải vừa một GPU. Dùng phiên sạch, không để
worker dùng phiên bản tracker cũ ghi cùng CSV.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import csv
import json
import math
import time
import shutil
from datetime import datetime

REPOSITORY_URL = "https://github.com/lngphgthao/moe-vihsd.git"
BRANCH = "main"  # Đổi nếu code mới nằm ở nhánh khác; phải push trước khi clone.
REPO_DIR = Path("/kaggle/working/code")
if Path("train.py").is_file():
    REPO_DIR = Path.cwd()
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPOSITORY_URL, str(REPO_DIR)], check=True)
assert (REPO_DIR / "tests/test_tracker_concurrency.py").is_file(), "Cần checkout code mới có tracker concurrency fix"
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
# Dự phòng khi Kaggle proxy/endpoint không hỗ trợ Xet; tải weights bằng HTTP chuẩn.
os.environ["HF_HUB_DISABLE_XET"] = "1"

## Secrets và output

Thêm HF_TOKEN (khuyến nghị) và WANDB_API_KEY trong Kaggle Secrets.
USE_WANDB=True yêu cầu API key; không in giá trị secrets. Chọn False chỉ khi chủ động
kiểm tra local/tracker mà chưa xác nhận W&B online. Output được giữ trong /kaggle/working.

In [ ]:
USE_WANDB = True
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for key in ("HF_TOKEN", "WANDB_API_KEY"):
    if not os.environ.get(key):
        try:
            os.environ[key] = secrets.get_secret(key)
        except Exception:
            print(f"Chưa lấy được {key}; thêm secret nếu cần")
assert not USE_WANDB or os.environ.get("WANDB_API_KEY"), "Thiếu WANDB_API_KEY"

OUTPUT = Path("/kaggle/working/parallel_smoke")
RUN_ROOT = OUTPUT / "runs"
LOG_ROOT = OUTPUT / "launcher_logs"
for folder in (OUTPUT, RUN_ROOT, LOG_ROOT):
    folder.mkdir(parents=True, exist_ok=True)
COMMON_ENV = {
    **os.environ,
    "HF_HUB_DISABLE_XET": "1",
    "HF_HOME": "/kaggle/working/hf_shared",
    "HF_HUB_CACHE": "/kaggle/working/hf_shared/hub",
    "HF_DATASETS_CACHE": "/kaggle/working/hf_shared/datasets",
    "RUNS_DIR": str(RUN_ROOT),
    "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
    "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1",
}
# Set before importing HF modules in this kernel.
os.environ.update({key: COMMON_ENV[key] for key in ("HF_HOME", "HF_HUB_CACHE", "HF_DATASETS_CACHE")})

## Preflight: hai GPU và test tracker offline

Cell này chưa chạy training. Nếu test lock/metadata thất bại thì dừng trước bước launch.

In [ ]:
gpu_text = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"], text=True)
print(gpu_text)
assert len(gpu_text.strip().splitlines()) >= 2, "Cần bật T4 x2"
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests",
                "-p", "test_tracker_concurrency.py", "-v"],
               env={**COMMON_ENV, "HF_HUB_OFFLINE": "1"}, check=True)

## Hai run và đăng ký tracker

run-014/015 đã giữ chỗ trong repo. Nếu đã dùng thì đổi sang ID chưa dùng; không sửa
trạng thái completed/failed về planned để chạy lại. File .lock có thể tồn tại sau khi
nhả khóa, không được xóa nó trong lúc các worker đang chạy.

In [ ]:
from src.utils import load_config
from src.utils.artifacts import TRACKER, reserve_run

JOBS = [
    {"gpu": "0", "run_id": "run-014", "config": "configs/diagnostics/vianli_xlmr_parallel_smoke.yaml"},
    {"gpu": "1", "run_id": "run-015", "config": "configs/diagnostics/vianli_mdeberta_parallel_smoke.yaml"},
]
assert len({job["run_id"] for job in JOBS}) == 2
for job in JOBS:
    assert not (RUN_ROOT / job["run_id"]).exists(), f"Artifact đã có: {job['run_id']}"
    assert not (LOG_ROOT / f"{job['run_id']}.log").exists(), "Launcher log đã có; dùng ID mới"
    config = load_config(job["config"])
    assert config["logging"]["use_tracker"] and config["training"]["smoke_test"]
    job["experiment_id"] = config["experiment_id"]
    job["backbone"] = config["model"]["backbone"]
    reserve_run(config, job["run_id"], tracker=TRACKER)
shutil.copy2(TRACKER, OUTPUT / "tracker_before.csv")
print([{**job, "backbone": load_config(job["config"])["model"]["backbone"]} for job in JOBS])

## Làm ấm cache dùng chung

Tải dữ liệu/tokenizer/weights tuần tự trước để thời gian tải không che mất việc hai
GPU training chồng nhau. Không dựng model hoặc optimizer ở cell này. Chỉ lấy weights
PyTorch/Safetensors cần thiết, không tải generator/TensorFlow. Dung lượng tải vài GB.

In [ ]:
from huggingface_hub import HfApi, snapshot_download
from src.data.loader import _load_raw
seen_models, seen_data = set(), set()
for job in JOBS:
    config = load_config(job["config"])
    model = config["model"]
    key = (model["backbone"], model["revision"])
    if key not in seen_models:
        info = HfApi().model_info(key[0], revision=key[1])
        files = [file.rfilename for file in info.siblings]
        safe = [name for name in files if name == "model.safetensors" or name == "model.safetensors.index.json"
                or (name.startswith("model-") and name.endswith(".safetensors"))]
        weights = safe or [name for name in files if name == "pytorch_model.bin" or name == "pytorch_model.bin.index.json"
                           or (name.startswith("pytorch_model-") and name.endswith(".bin"))]
        assert weights, f"Không tìm thấy pretrained weights: {key[0]}"
        snapshot_download(key[0], revision=key[1], allow_patterns=weights + [
            "config.json", "tokenizer*", "special_tokens_map.json", "added_tokens.json",
            "*.model", "vocab*", "merges.txt", "bpe.codes"], max_workers=2)
        seen_models.add(key)
        print("Model cache ready:", key[0])
    data_key = (config["dataset"]["name"], config["dataset"].get("revision"))
    if data_key not in seen_data:
        dataset = _load_raw(config["dataset"])
        print("Dataset cache ready:", {name: len(split) for name, split in dataset.items()})
        seen_data.add(data_key)

## Launch: mỗi process chỉ nhìn thấy một GPU

W&B run name là run_id; tracker và output root dùng chung. Bộ nhớ model/optimizer/RNG
thuộc từng process. Nếu cell launch lỗi giữa chừng, chỉ các process vừa khởi tạo bởi
cell này được dừng; không đụng process khác.

In [ ]:
def stop_workers(processes):
    for process in processes.values():
        if process.poll() is None:
            process.terminate()
    for process in processes.values():
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()


def launch_jobs(jobs):
    processes = {}
    try:
        for job in jobs:
            env = {**COMMON_ENV, "CUDA_VISIBLE_DEVICES": job["gpu"]}
            args = [sys.executable, "train.py", "--config", job["config"],
                    "--run-id", job["run_id"], "--smoke-test",
                    "--set", "logging.use_tracker=true", "--set", f"logging.use_wandb={str(USE_WANDB).lower()}"]
            with (LOG_ROOT / f"{job['run_id']}.log").open("x", encoding="utf-8") as log:
                process = subprocess.Popen(args, cwd=REPO_DIR, env=env,
                                           stdout=log, stderr=subprocess.STDOUT)
            processes[job["run_id"]] = process
            print(f"{job['run_id']}: physical GPU {job['gpu']}, PID {process.pid}")
    except BaseException:
        stop_workers(processes)
        raise
    return processes


PROCESSES = launch_jobs(JOBS)

## Quan sát tới khi cả hai process kết thúc

Mỗi 10 giây xem GPU utilization/memory và phần cuối log. Lúc CPU chuẩn bị model,
GPU utilization có thể chưa tăng. Cả hai process có thể ghi device=cuda; đọc thêm
hardware.cuda_visible_devices để phân biệt GPU vật lý. None=đang chạy, 0=thành công.
Interrupt cell này sẽ dừng các worker của notebook; retry cần ID mới.

In [ ]:
GPU_QUERY = ["nvidia-smi", "--query-gpu=index,name,utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"]
try:
    while True:
        states = {run_id: process.poll() for run_id, process in PROCESSES.items()}
        sample = subprocess.run(GPU_QUERY, capture_output=True, text=True)
        with (OUTPUT / "gpu_samples.jsonl").open("a", encoding="utf-8") as stream:
            stream.write(json.dumps({"time": datetime.now().isoformat(), "states": states,
                                     "gpu": sample.stdout.strip(), "query_exit_code": sample.returncode}) + "\n")
        print(states, "\n", sample.stdout.strip())
        for job in JOBS:
            path = LOG_ROOT / f"{job['run_id']}.log"
            print(job["run_id"], "\n", "\n".join(path.read_text(errors="replace").splitlines()[-3:]))
        if all(code is not None for code in states.values()):
            break
        time.sleep(10)
except KeyboardInterrupt:
    stop_workers(PROCESSES)
    raise
EXIT_CODES = {run_id: process.wait() for run_id, process in PROCESSES.items()}
print("Exit codes:", EXIT_CODES)

## Xác nhận bằng artifact, GPU assignment và tracker

PASS cần cả hai completed/smoke, GPU mask khác nhau, có khoảng thời gian train/validation
chồng nhau, checkpoint/metrics riêng, CSV đủ hai dòng đúng metadata và W&B synced nếu bật.
Nếu overlap=false thì chưa xác nhận hai training window chạy song song dù cả hai có thể
hoàn tất riêng lẻ; xem log/cache, tăng budget diagnostic và cấp ID mới trước retry.

In [ ]:
def verify_runs(jobs, codes, tracker_path, run_root, require_wandb):
    with Path(tracker_path).open(encoding="utf-8", newline="") as stream:
        records = list(csv.DictReader(stream))
    reports, windows = {}, []
    for job in jobs:
        run_id = job["run_id"]
        folder = Path(run_root) / run_id
        missing = [name for name in ("manifest.json", "summary.json", "metrics.jsonl") if not (folder / name).exists()]
        if missing:
            reports[run_id] = {"checks": {"required_artifacts": False, "exit_zero": codes[run_id] == 0}, "missing": missing}
            continue
        manifest = json.loads((folder / "manifest.json").read_text())
        summary = json.loads((folder / "summary.json").read_text())
        metrics = [json.loads(line) for line in (folder / "metrics.jsonl").read_text().splitlines() if line.strip()]
        matches = [row for row in records if row["run_id"] == run_id]
        assert len(matches) == 1, f"CSV thiếu/trùng dòng {run_id}"
        row = matches[0]
        hardware = manifest["hardware"]
        checkpoint = folder / summary["checkpoint"]
        checks = {
            "exit_zero": codes[run_id] == 0,
            "completed_smoke": manifest["run_id"] == run_id and manifest["status"] == "completed" and manifest["profile"] == "smoke" and manifest["csv_tracker_enabled"],
            "one_gpu_correct_mask": hardware["device"] == "cuda" and hardware["visible_gpu_count"] == 1 and hardware["cuda_visible_devices"] == job["gpu"],
            "tracker_matches": row["status"] == "completed" and row["dataset"] == "vianli" and row["backbone"] == manifest["model"]["backbone"]
                and math.isclose(float(row["val_macro_f1"]), summary["validation"]["macro_f1"], abs_tol=1e-9),
            "expected_model": manifest["experiment_id"] == job["experiment_id"] and manifest["model"]["backbone"] == job["backbone"],
            "checkpoint_in_own_run": checkpoint.is_file() and checkpoint.resolve().is_relative_to(folder.resolve()),
            "train_val_only": {record["split"] for record in metrics} == {"train", "validation"} and not (folder / "evaluation/test.json").exists(),
            "finite_losses": bool(metrics) and all(math.isfinite(record["loss"]) for record in metrics),
            "wandb_synced": not require_wandb or (manifest["wandb_sync"] == "synced" and bool(manifest.get("wandb_url"))),
        }
        windows.append((datetime.fromisoformat(manifest["training_started_at"]), datetime.fromisoformat(manifest["training_finished_at"])))
        reports[run_id] = {"checks": checks, "wandb_url": manifest.get("wandb_url"),
                           "best_epoch": summary["best_epoch"], "validation": summary["validation"]}
    overlap_seconds = (min(end for _, end in windows) - max(start for start, _ in windows)).total_seconds() if len(windows) == len(jobs) else 0.0
    return {"runs": reports, "training_overlap_seconds": max(0.0, overlap_seconds),
            "passed": overlap_seconds > 0 and all(all(report["checks"].values()) for report in reports.values())}


# Keep artifacts/tracker even if verification fails.
shutil.copy2(TRACKER, OUTPUT / "tracker_after.csv")
report = verify_runs(JOBS, EXIT_CODES, TRACKER, RUN_ROOT, USE_WANDB)
(OUTPUT / "confirmation.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
assert report["passed"], "Chưa xác nhận đầy đủ; xem confirmation.json, launcher logs và GPU samples"

## Lưu output

Kết quả local ở /kaggle/working/parallel_smoke/: runs/, launcher_logs/, tracker_before.csv,
tracker_after.csv, gpu_samples.jsonl và confirmation.json. Dùng Save Version để giữ output;
không chỉ tải checkpoint. Có thể tạo ZIP bằng cell cuối. Đây là diagnostic, không gọi
lệnh evaluate test hoặc dùng score nhỏ này để chọn backbone/finalist.

In [ ]:
CREATE_ZIP = False  # Bật nếu cần một file tải về; hai checkpoints có thể vài GB.
if CREATE_ZIP:
    archive = shutil.make_archive(str(OUTPUT.parent / "parallel_smoke_output"), "zip", root_dir=OUTPUT)
    print("Download:", archive)
else:
    print("Save Version để giữ toàn bộ output:", OUTPUT)
